# State 60482 — welches Paper erklärt den Effekt?

Eine Pile-Passage kam bei Optimizer-Schritt **131278** ins Training von Pythia-1.4B. Über die
Checkpoints, die diesen Schritt einrahmen, stieg die Wahrscheinlichkeit eines zurückgehaltenen
Zieltokens (`" per"`, in „74 km per hour") um relativ +14 %. Das wurde zunächst als
**Memorierung** gelesen.

Dieses Notebook prüft diese Lesart gegen die sechs Paper aus der Kandidatenliste — und gegen
drei Erklärungen, die nicht auf der Liste stehen.

**Aufbau.** Jede Hypothese wird zu einer Probe mit einer Entscheidungsregel, die *vor* den Daten
feststeht. Jede Probe vergleicht den Anker gegen 40 Kontrollpassagen und gegen zwei
Placebo-Grenzen. Jede Probe sagt am Ende, was sie **nicht** entscheiden kann.

**Reihenfolge.** Die billigen Proben laufen zuerst, weil sie die Prämisse kippen können:
Wenn der Greedy-Fortsetzung des Modells nie `" per"` ist, ist die Passage unter keiner
publizierten Definition memoriert, und die teuren mechanistischen Proben beantworten eine
Frage, die niemand stellen sollte.

**Laufzeit.** Etwa 35–50 Minuten auf einer A100-40GB, davon der größte Teil Download der vier
Checkpoints (~23 GB in fp32). Die Messung selbst ist billig.

---

**Runtime setzen:** `Laufzeit → Laufzeittyp ändern → A100 GPU`.

## 1. Installation und Code

Das Paket liegt im Repository unter `macinterp_60482/`. Ohne Repository-Zugriff greift die
Zelle auf einen Klon zurück.

In [ ]:
# 1. Setup
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/assistant-axis")
REPO_URL = "https://github.com/erikiss/assistant-axis.git"
BRANCH = "claude/macinterp-60482-state-23kvi0"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

PKG = REPO_DIR / "macinterp_60482"
sys.path.insert(0, str(PKG))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.40", "accelerate", "huggingface_hub", "numpy", "pandas"], check=True)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 1e9:.0f} GB")
    if p.total_memory < 30e9:
        print("WARNUNG: weniger als 30 GB. fp32-Pythia-1.4B passt, aber ohne Reserve.")
else:
    print("Keine GPU. Nur --smoke moeglich.")

## 2. Die Passagen

Der kanonische K8-Satz hat 69 Einträge: 19 States (darunter `state_60482`, T = 207), 40
Kontrollen und 10 nie trainierte Namensvarianten. Er wird über den kanonischen SHA-256 geprüft.

**Vier Herkunftsstufen**, absteigend nach Vertrauen:

| Stufe | Bedeutung | Ergebnisse vergleichbar? |
|---|---|---|
| `canonical` | Hash stimmt mit dem K8-Snapshot überein | ja |
| `regenerated` | aus den Exp-2-Tabellen neu erzeugt, Hash stimmt | ja |
| `reconstructed` | Struktur stimmt, Hash nicht — Kontrollen neu gezogen | **nein** |
| `synthetic` | bedeutungslos, nur zum Testen der Pipeline | **nein** |

Alles unterhalb von `regenerated` markiert jedes nachgelagerte Ergebnis dauerhaft.

In [ ]:
# 2. Passagen aufloesen
from google.colab import drive
try:
    drive.mount("/content/drive", force_remount=False)
    ROOTS = ["/content/drive/MyDrive/Colab_Pythia_Results", "/content/pythia_attn_runtime", "/content"]
except Exception as exc:
    print("Drive nicht gemountet:", exc)
    ROOTS = ["/content"]

from m60482 import passages as P

PASSAGEN_PATH = ""      # expliziter Pfad hat immer Vorrang
ALLOW_SYNTHETIC = False  # True nur, um die Pipeline ohne echte Daten zu testen

ps = P.resolve(PASSAGEN_PATH or None, ROOTS, allow_synthetic=ALLOW_SYNTHETIC)
print()
print(f"{len(ps.passages)} Passagen | {len(ps.states)} States, {len(ps.controls)} Kontrollen, {len(ps.variants)} Varianten")
print("Herkunft:", ps.provenance, "| vergleichbar:", ps.trustworthy)

## 3. Hauptmessung

Ein Forward-Pass pro (Checkpoint × Passage). Gespeichert werden:

- die 50 wahrscheinlichsten Fortsetzungen mit Wahrscheinlichkeiten — **die Rangliste selbst**,
  die im vorigen Lauf gefehlt hat
- Rang und Wahrscheinlichkeit des Ziels über das *volle* Vokabular
- die Attention der letzten Position auf jede Kontextposition je Schicht
- die über alle Anfragen gemittelte empfangene Aufmerksamkeit
- Gewichte je Kopf für die Anker-Familie
- die Kontext-NLL je Position, damit die Tokenebene-Auswertung aus demselben Bundle
  nachprüfbar ist

**fp32, nicht fp16.** Bei step131000 liegen Ziel (0.09087) und Konkurrent `"/"` (0.08965)
0.00122 auseinander. Die Schlagzeilen-Statistik ist ein *Rang*; in fp16 ist diese Ordnung nicht
reproduzierbar.

**`attn_implementation="eager"`.** Unter SDPA gibt `output_attentions=True` stillschweigend
`None` zurück. Dann fehlen die Attention-Zahlen, statt falsch zu sein — was schlimmer ist.

In [ ]:
# 3. Messen
import time
from m60482 import config as C
from m60482.measure import measure, verify_against_reference

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUT = Path("/content/m60482_runs") / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

cfg = C.RunConfig(
    checkpoints=C.CHECKPOINTS,     # 130000, 131000, 132000, 133000
    dtype="float32",
    device="cuda",
    top_k=50,
    capture_attention=True,
    run_id=RUN_ID,
    output_dir=str(OUT),
)

BUNDLE = OUT / "bundle.npz"
if BUNDLE.exists():
    from m60482.measure import Bundle
    bundle = Bundle.load(BUNDLE)
    print("bestehendes Bundle geladen:", BUNDLE)
else:
    bundle = measure(ps, cfg)
    bundle.save(BUNDLE)
    print("Bundle geschrieben:", BUNDLE)

### Gegenprobe gegen die frühere Messung

Stimmen die neuen Zahlen nicht mit den alten überein, misst dieser Lauf ein anderes Objekt —
falsche Modellvariante, falscher dtype, falscher Passagensatz — und alle Proben darunter
handeln von diesem anderen Objekt.

Ein Punkt, den diese Prüfung **nicht** klärt: `global_sample_index` ist suite-spezifisch.
`pythia-1.4b` und `pythia-1.4b-deduped` haben verschiedene Datenreihenfolgen, also bezeichnet
Index 134428942 in beiden eine andere Sequenz. Welche Suite den Anker erzeugt hat, entscheidet
die Architektur nicht.

In [ ]:
# 3b. Reproduktionspruefung
import pandas as pd

check = verify_against_reference(bundle)
if check.get("checked"):
    print("stimmt mit der frueheren Messung ueberein:", check["all_ok"])
    display(pd.DataFrame(check["rows"]))
    if not check["all_ok"]:
        print("\nACHTUNG:", check["note"])
print()
print(C.DEDUP_AMBIGUITY)

## 4. Zusatzmessungen

Drei Fragen, die ein Forward-Pass pro Passage nicht beantwortet:

**Wie viel Kontext braucht die Vorhersage wirklich?**
Die Trunkierungsleiter misst p(Ziel) mit nur den letzten *k* Tokens. Liegt der Wert bei k = 2
schon auf dem Vollkontext-Wert, dann tragen die übrigen 205 Tokens, der Name und die Exposition
nichts zu der gemessenen Zahl bei — und jede passagenbezogene Hypothese erklärt etwas, das es
nicht gibt.

**Würde das Modell den Trainingstext überhaupt wiedergeben?**
Jede publizierte Definition von wortgetreuer Memorierung ist über die *Greedy*-Fortsetzung
formuliert. Das ist das Eintrittskriterium, und es kostet vier kurze Generierungen.

**Trägt der Name?**
Die Namenstokens werden an Ort und Stelle ersetzt — das misst den kausalen Beitrag der Stelle,
von der die Memorierungs-Geschichte abhängt.

**Und der untrainierte Vergleich.** Das einzige Paper der Liste, dessen Voraussetzungen hier
erfüllt sind, macht seine Aussage über *untrainierte* Netze. Ohne diese Basislinie ist
„alle 69 Passagen teilen ein Profil" genauso gut mit „alle haben dasselbe gelernt" verträglich.
Kostet einen Sweep und keinen Download.

In [ ]:
# 4. Zusatzmessungen
from m60482 import aux as A

AUX = OUT / "aux.npz"
RUN_UNTRAINED = True   # eine zusaetzliche Sweep-Runde, kein Download

if "ladder" in (bundle.aux or {}):
    print("Zusatzmessungen bereits im Bundle.")
else:
    extra = A.measure_aux(ps, cfg)
    if RUN_UNTRAINED:
        extra.update(A.untrained_attention(ps, cfg))
    bundle.aux.update(extra)
    bundle.save(BUNDLE)
    print("Bundle mit Zusatzmessungen aktualisiert.")

### Attention ist keine Attribution

Das auffälligste Attention-Ergebnis ist ein *Null*: Der Anker verschiebt sich über die
Expositionsgrenze **weniger** als 40 von 40 Kontrollen. Das wurde als „kein Expositionssignal
in der Attention" gelesen.

Dafür fehlt noch eine Messung. In den Residualstrom fließt `α_j · v_j`, und Sink-Positionen
sind genau die mit ausgetrockneten Value-Vektoren — so funktioniert ein Sink als No-Op. Eine
Position kann 0.69 der Masse halten und fast nichts beitragen. Ein Null in einer Größe, die
keinen Beitrag misst, ist in beide Richtungen uninformativ.

Die folgende Zelle rechnet das Profil als `|α_j| · ‖v_j‖` neu (Kobayashi et al.,
arXiv:2004.10102). Sie kostet einen zweiten Forward-Pass je Passage und läuft deshalb nur an
den beiden Checkpoints, die die Expositionsgrenze einrahmen.

In [ ]:
# 4c. Norm-gewichtete Attention (nur an der Expositionsgrenze)
if "norm_attn" in (bundle.aux or {}):
    print("Norm-gewichtete Attention bereits im Bundle.")
else:
    bundle.aux.update(A.measure_norm_attribution(ps, cfg))
    bundle.save(BUNDLE)
    print("Bundle aktualisiert.")

# roh gegen norm-gewichtet an den dominanten Positionen
norm = np.asarray(bundle.aux["norm_attn"])
nsteps = list(bundle.aux["norm_attn_steps"])
nnames = list(bundle.aux["norm_attn_names"])
s_i, na_i = len(nsteps) - 1, nnames.index(C.ANCHOR)

raw = bundle.attn_last[bundle.ci(nsteps[-1]), bundle.pi(C.ANCHOR), 3:].mean(axis=0)
nrm = norm[s_i, na_i, 3:].mean(axis=0)
ctx_tokens = bundle.context_tokens.get(C.ANCHOR, [])
top = np.argsort(-raw)[:4]

display(pd.DataFrame([
    {"Position": int(p),
     "Token": repr(ctx_tokens[p]) if p < len(ctx_tokens) else f"pos{p}",
     "roh": float(raw[p]),
     "norm-gewichtet": float(nrm[p]),
     "Anteil erhalten": float(nrm[p] / raw[p]) if raw[p] > 0 else float("nan")}
    for p in top
]).round(5))

In [ ]:
# 4b. Die Trunkierungsleiter ansehen -- das ist die entscheidende Tabelle
import numpy as np

ladder = np.asarray(bundle.aux["ladder"])
ks = bundle.aux["ladder_k"]
lnames = bundle.aux["ladder_names"]
step = C.EXPOSURE_BOUNDARY[1]
ci, ai = bundle.ci(step), lnames.index(C.ANCHOR)

full = float(ladder[ci, ai, -1])
display(pd.DataFrame([
    {"k": k, "p(' per')": float(ladder[ci, ai, i]), "Anteil am Vollkontext": float(ladder[ci, ai, i] / full)}
    for i, k in enumerate(ks)
]).round(5))
print(f"Vollkontext (k=207): {full:.5f}")

## 5. Proben

Zwölf Proben, nach Kosten sortiert. Jede liefert ein Urteil aus
`SUPPORTED / REFUTED / INCONCLUSIVE / SCOPE_FAILED / NOT_RUN / ERROR` gegen ihre
vorregistrierte Entscheidungsregel.

`SCOPE_FAILED` heißt nicht, dass ein Paper falsch ist. Es heißt, dass sein Gegenstand hier
nicht vorkommt.

In [ ]:
# 5. Proben laufen lassen
from m60482 import registry, report
import m60482.probes  # registriert die Proben

results = registry.run(bundle)

In [ ]:
# 5b. Urteil je Paper -- die eigentliche Frage
adj = report.adjudicate(results)
print(adj["headline"])
print()

display(pd.DataFrame([
    {"Paper": p["key"], "Zitat": p["citation"], "Urteil": p["verdict"],
     "Proben": ", ".join(f"{k}={v}" for k, v in p["probes"].items())}
    for p in adj["papers"] if p["on_user_list"] and p["kind"] == "explanation"
]))

print("\nNicht auf der Liste:")
display(pd.DataFrame([
    {"Kandidat": p["key"], "Zitat": p["citation"], "Urteil": p["verdict"]}
    for p in adj["papers"] if not p["on_user_list"] and p["kind"] == "explanation"
]))

print("\nPraemissen-Pruefungen (keine Erklaerungen):")
display(pd.DataFrame([
    {"Pruefung": k, "Urteil": v}
    for p in adj["papers"] if p["kind"] == "premise" for k, v in p["probes"].items()
]))

In [ ]:
# 5c. Vollstaendiger Bericht
paths = report.save(bundle, results, OUT)
print("Bericht:", paths["report"])
print("Rohdaten:", paths["results"])

from IPython.display import Markdown, display as d
d(Markdown(paths["report"].read_text()))

## 6. Was dieser Lauf grundsätzlich nicht kann

Unabhängig vom Ergebnis:

- Mit 40 Kontrollen ist der kleinste erreichbare p-Wert **1/41 = 0.0244**. Nach Korrektur für
  die Zahl der Proben ist hier kein Ergebnis signifikant. Das ist keine Eigenschaft der Daten,
  sondern des Designs.
- Es gibt **drei** Grenzen, eine davon ist die Expositionsgrenze. Die beiden Placebo-Grenzen
  sind die gesamte Nullverteilung — „außerhalb des Placebo-Bereichs" heißt „außerhalb eines
  aus zwei Zahlen geschätzten Bereichs".
- Der Anker wurde ausgewählt, **weil** er auffällig war. Jede Statistik, die eine Funktion des
  Auswahlkriteriums ist, ist beschreibend; jede Probe sagt, auf welcher Seite dieser Linie sie
  steht.
- Attention zeigt, welche Schicht welche Stelle liest. Sie zeigt **keine Reihenfolge**. Ein
  Transformer hat Tiefe, keine Zeit.
- Aufmerksamkeitsmasse ist keine Attribution. Ein Kopf mit 2 % der Masse kann die
  entscheidende Information tragen. Das zu klären bräuchte Ablation oder Patching — das macht
  keine dieser Proben.
- Vier Checkpoints sind vier Stichproben aus einem verrauschten Prozess. Ein Effekt, der an
  einem einzigen Checkpoint hängt und nicht persistent ist, bleibt genau das, auch wenn eine
  Probe ihn bestätigt.